In [1]:
!pip install -q \
    openai \
    pymupdf \
    chromadb \
    sentence-transformers \
    requests \
    gradio \
    nest_asyncio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-cloud-bigquery 3.47.0 requi

In [2]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
print("API key configured.")

Enter your OpenAI API key: ··········
API key configured.


In [3]:
# ============================================================
# ResearchPilot – Clean Academic Research Assistant (Colab)
# ============================================================
# Features:
#   • Upload & index PDF papers (local RAG with Chroma)
#   • External literature search via OpenAlex
#   • Combined or mode-specific research
#   • Strict citation discipline (local text vs metadata)
#   • Gradio UI
# ============================================================

import os
import re
import hashlib
from pathlib import Path
from typing import List, Dict, Any, Optional

import gradio as gr
import pymupdf
import chromadb
import requests
from openai import OpenAI
from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
BASE_DIR = Path("/content/researchpilot")
DB_DIR = BASE_DIR / "chroma_db"
DB_DIR.mkdir(parents=True, exist_ok=True)

COLLECTION_NAME = "research_papers"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
LLM_MODEL = "gpt-4o-mini"
MAX_PDF_PAGES = 200
MAX_PDF_SIZE_MB = 25
CHUNK_WORDS = 450
CHUNK_OVERLAP = 50

# ------------------------------------------------------------
# Global objects (loaded once)
# ------------------------------------------------------------
client = OpenAI()
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
chroma_client = chromadb.PersistentClient(path=str(DB_DIR))
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

print(f"ResearchPilot ready. Indexed chunks: {collection.count()}")

# ------------------------------------------------------------
# PDF Indexing
# ------------------------------------------------------------
def index_pdf(file_path: Optional[str]) -> str:
    if not file_path:
        return "Please select a PDF file."

    path = Path(file_path)
    if path.suffix.lower() != ".pdf":
        return "Only PDF files are supported."

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > MAX_PDF_SIZE_MB:
        return f"PDF exceeds {MAX_PDF_SIZE_MB} MB limit ({size_mb:.1f} MB)."

    try:
        with pymupdf.open(str(path)) as doc:
            if doc.is_encrypted:
                return "Encrypted PDFs are not supported."
            if len(doc) > MAX_PDF_PAGES:
                return f"PDF exceeds {MAX_PDF_PAGES}-page limit."

            file_hash = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
            documents, metadatas, ids = [], [], []

            for page_num, page in enumerate(doc, start=1):
                text = page.get_text().strip()
                if not text:
                    continue

                words = text.split()
                start = 0
                chunk_idx = 0
                while start < len(words):
                    end = min(start + CHUNK_WORDS, len(words))
                    chunk = " ".join(words[start:end]).strip()
                    if chunk:
                        documents.append(chunk)
                        metadatas.append({
                            "paper": path.name,
                            "page": page_num,
                            "file_hash": file_hash
                        })
                        ids.append(f"{file_hash}_p{page_num}_c{chunk_idx}")
                        chunk_idx += 1
                    if end == len(words):
                        break
                    start = end - CHUNK_OVERLAP

            if not documents:
                return "No extractable text found. The PDF may be image-only (OCR required)."

            embeddings = embedder.encode(
                documents, normalize_embeddings=True, show_progress_bar=False
            ).tolist()

            collection.upsert(
                ids=ids,
                documents=documents,
                metadatas=metadatas,
                embeddings=embeddings
            )

            return (
                f"✓ Indexed {len(documents)} chunks from **{path.name}**. "
                f"Total chunks in database: {collection.count()}"
            )

    except Exception as e:
        return f"PDF indexing failed: {type(e).__name__}: {e}"


# ------------------------------------------------------------
# Local Retrieval (RAG)
# ------------------------------------------------------------
def retrieve_local(question: str, k: int = 5) -> List[Dict[str, Any]]:
    if not question or not question.strip():
        return []

    count = collection.count()
    if count == 0:
        return []

    query_emb = embedder.encode(question, normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=min(k, count),
        include=["documents", "metadatas", "distances"]
    )

    evidence = []
    for text, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        evidence.append({
            "text": text,
            "paper": meta["paper"],
            "page": meta["page"],
            "distance": float(dist)
        })
    return evidence


# ------------------------------------------------------------
# External Search (OpenAlex)
# ------------------------------------------------------------
def search_openalex(topic: str, limit: int = 5) -> List[Dict[str, Any]]:
    topic = (topic or "").strip()
    if not topic:
        return []

    limit = max(1, min(int(limit), 15))
    params = {
        "search": topic,
        "per-page": limit,
        "filter": "from_publication_date:2018-01-01"
    }
    email = os.getenv("OPENALEX_EMAIL")
    if email:
        params["mailto"] = email

    try:
        r = requests.get(
            "https://api.openalex.org/works",
            params=params,
            timeout=25
        )
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f"OpenAlex error: {e}")
        return []

    papers = []
    for work in data.get("results", []):
        authors = []
        for authorship in work.get("authorships", [])[:6]:
            name = authorship.get("author", {}).get("display_name")
            if name:
                authors.append(name)

        papers.append({
            "title": work.get("display_name") or "Untitled",
            "year": work.get("publication_year"),
            "authors": authors,
            "doi": work.get("doi"),
            "openalex_id": work.get("id"),
            "cited_by_count": work.get("cited_by_count", 0)
        })
    return papers


# ------------------------------------------------------------
# Evidence formatting & citation helpers
# ------------------------------------------------------------
def format_local_evidence(local: List[Dict]) -> List[str]:
    blocks = []
    for i, item in enumerate(local, 1):
        blocks.append(
            f"[L{i}] Local PDF – {item['paper']}, page {item['page']}\n"
            f"{item['text']}"
        )
    return blocks


def format_external_evidence(papers: List[Dict]) -> List[str]:
    blocks = []
    for i, p in enumerate(papers, 1):
        authors = ", ".join(p["authors"]) if p["authors"] else "Unknown"
        blocks.append(
            f"[E{i}] External (metadata only)\n"
            f"Title: {p['title']}\n"
            f"Year: {p['year']}\n"
            f"Authors: {authors}\n"
            f"DOI: {p['doi'] or 'N/A'}\n"
            f"Citations: {p['cited_by_count']}\n"
            f"OpenAlex: {p['openalex_id']}\n"
            "NOTE: Only bibliographic metadata is available. "
            "Do not claim specific findings from this paper."
        )
    return blocks


def validate_citations(answer: str, available_ids: List[str]) -> Dict[str, Any]:
    cited = set(re.findall(r"\[([LE]\d+)\]", answer))
    available = set(available_ids)
    missing = cited - available
    malformed = re.findall(r"\[(\d+)\]", answer)  # bare numbers

    return {
        "valid": len(missing) == 0 and len(malformed) == 0 and len(cited) > 0,
        "cited": sorted(cited),
        "available": sorted(available),
        "missing": sorted(missing),
        "malformed": malformed
    }


# ------------------------------------------------------------
# Core research function
# ------------------------------------------------------------
def research(question: str, mode: str = "Combined") -> str:
    question = (question or "").strip()
    if not question:
        return "Please enter a research question."

    if not os.getenv("OPENAI_API_KEY"):
        return "OPENAI_API_KEY is not set. Please configure it first."

    evidence_blocks = []
    evidence_ids = []

    # Local RAG
    if mode in ("Local PDF", "Combined"):
        local = retrieve_local(question, k=6)
        for i, item in enumerate(local, 1):
            eid = f"L{i}"
            evidence_ids.append(eid)
            evidence_blocks.append(
                f"[{eid}] Local PDF – {item['paper']}, page {item['page']}\n"
                f"{item['text']}"
            )

    # External OpenAlex
    if mode in ("External Papers", "Combined"):
        papers = search_openalex(question, limit=6)
        for i, p in enumerate(papers, 1):
            eid = f"E{i}"
            evidence_ids.append(eid)
            authors = ", ".join(p["authors"]) if p["authors"] else "Unknown"
            evidence_blocks.append(
                f"[{eid}] External (metadata only)\n"
                f"Title: {p['title']}\n"
                f"Year: {p['year']}\n"
                f"Authors: {authors}\n"
                f"DOI: {p['doi'] or 'N/A'}\n"
                f"Citations: {p['cited_by_count']}\n"
                f"OpenAlex: {p['openalex_id']}\n"
                "NOTE: Only bibliographic metadata is available. "
                "Do not claim specific findings from this paper."
            )

    if not evidence_blocks:
        return (
            "No evidence found.\n\n"
            "• Upload and index at least one PDF for Local / Combined mode.\n"
            "• Or try a broader query for External Papers mode."
        )

    evidence_text = "\n\n".join(evidence_blocks)

    system_prompt = (
        "You are ResearchPilot, a rigorous academic research assistant.\n\n"
        "Rules:\n"
        "1. Base every factual claim strictly on the supplied evidence.\n"
        "2. Cite local text evidence with [L1], [L2], etc.\n"
        "3. Cite external sources with [E1], [E2], etc.\n"
        "4. External sources currently provide only bibliographic metadata. "
        "You may note them as potentially relevant literature, but you must "
        "NOT claim that they support specific scientific findings.\n"
        "5. Never invent authors, titles, results, page numbers, or quotations.\n"
        "6. If the evidence is insufficient, state the limitation clearly.\n"
        "7. Write clear, MSc-level academic prose. Use short headings when helpful."
    )

    user_prompt = (
        f"Research question:\n{question}\n\n"
        f"Available evidence:\n{evidence_text}"
    )

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.2,
            timeout=60
        )
        answer = response.choices[0].message.content or ""
    except Exception as e:
        return f"LLM request failed: {type(e).__name__}: {e}"

    # Citation validation
    check = validate_citations(answer, evidence_ids)
    warning = ""
    if not check["valid"]:
        warning = (
            "\n\n---\n"
            "**Citation warning:** Some citations may be missing or malformed. "
            f"Cited: {check['cited']} | Available: {check['available']} | "
            f"Missing: {check['missing']}"
        )

    references = "\n\n".join(evidence_blocks)

    return (
        answer
        + warning
        + "\n\n---\n\n### Retrieved evidence\n\n"
        + references
    )


# ------------------------------------------------------------
# Gradio UI
# ------------------------------------------------------------
with gr.Blocks(
    title="ResearchPilot",
    theme=gr.themes.Soft()
) as demo:
    gr.Markdown(
        """
        # ResearchPilot
        Academic research assistant with **local PDF retrieval (RAG)** and **OpenAlex literature discovery**.
        """
    )

    with gr.Tab("1. Upload & Index Papers"):
        gr.Markdown(
            "Upload one or more PDFs. Text is extracted, chunked, embedded, "
            "and stored in a local Chroma database."
        )
        pdf_input = gr.File(
            label="PDF file",
            file_types=[".pdf"],
            type="filepath"
        )
        index_btn = gr.Button("Index PDF", variant="primary")
        index_status = gr.Markdown()

        index_btn.click(
            fn=index_pdf,
            inputs=pdf_input,
            outputs=index_status
        )

        gr.Markdown(f"**Current indexed chunks:** `{collection.count()}`")

    with gr.Tab("2. Research"):
        question = gr.Textbox(
            label="Research Question",
            lines=4,
            placeholder="e.g. What are the main limitations of RAG systems discussed in the uploaded papers?"
        )
        mode = gr.Radio(
            choices=["Local PDF", "External Papers", "Combined"],
            value="Combined",
            label="Research Mode"
        )
        run_btn = gr.Button("Run Research", variant="primary")
        answer_box = gr.Markdown(label="Answer")

        run_btn.click(
            fn=research,
            inputs=[question, mode],
            outputs=answer_box
        )

    with gr.Tab("About"):
        gr.Markdown(
            """
            ### How it works
            - **Local PDF**: Semantic search over your uploaded papers + LLM synthesis with [L#] citations.
            - **External Papers**: OpenAlex search (metadata only) + careful synthesis with [E#] citations.
            - **Combined**: Both sources together.

            ### Important notes
            - External results currently provide **bibliographic metadata only**. The model is instructed not to invent findings from titles alone.
            - Always verify critical claims against the original papers.
            - The vector database persists inside `/content/researchpilot/chroma_db` for the lifetime of the Colab runtime.
            """
        )


# Launch
if __name__ == "__main__":
    demo.launch(share=True, debug=False, show_error=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ResearchPilot ready. Indexed chunks: 0


/tmp/ipykernel_15316/2323887544.py:363: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76df5ec4083a0242d8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
demo.launch(share=True, debug=False, show_error=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76df5ec4083a0242d8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
